In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


# LightGBM Baseline

This notebook trains a LightGBM multiclass classification model using the same cross-validation strategy as the previous baselines.

In [2]:
import pandas as pd

path = "/kaggle/input/competitions/playground-series-s6e7"

train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()


(690088, 15)
(295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [4]:
X = train.drop(columns=["health_condition", "id"])
y = train["health_condition"]

X_test = test.drop(columns=["id"])

print(X.shape)
print(y.shape)
print(X_test.shape)

(690088, 13)
(690088,)
(295753, 13)


# LightGBM Baseline

This notebook trains a LightGBM multiclass classification model using the same cross-validation strategy as the previous baselines.

In [5]:
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

In [6]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Original classes:", label_encoder.classes_)
print("Encoded classes:", np.unique(y_encoded))

Original classes: ['at-risk' 'fit' 'unhealthy']
Encoded classes: [0 1 2]


In [7]:
categorical_columns = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [8]:
for column in categorical_columns:
    combined = pd.concat(
        [X[column], X_test[column]],
        axis=0
    ).astype("category")

    X[column] = combined.iloc[:len(X)].reset_index(drop=True)
    X_test[column] = combined.iloc[len(X):].reset_index(drop=True)

print(X.dtypes)

sleep_duration              float64
heart_rate                  float64
bmi                         float64
calorie_expenditure         float64
step_count                  float64
exercise_duration           float64
water_intake                float64
diet_type                  category
stress_level               category
sleep_quality              category
physical_activity_level    category
smoking_alcohol            category
gender                     category
dtype: object


In [9]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions = np.zeros(len(X), dtype=int)

test_probabilities = np.zeros(
    (len(X_test), len(label_encoder.classes_))
)

fold_scores = []

In [10]:
for fold, (train_index, valid_index) in enumerate(
    skf.split(X, y_encoded),
    start=1
):
    print(f"\nFold {fold}")

    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]

    y_train = y_encoded[train_index]
    y_valid = y_encoded[valid_index]

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(label_encoder.classes_),
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="multi_logloss",
        categorical_feature=categorical_columns,
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            ),
            lgb.log_evaluation(100)
        ]
    )

    valid_predictions = model.predict(
        X_valid,
        num_iteration=model.best_iteration_
    )

    fold_score = balanced_accuracy_score(
        y_valid,
        valid_predictions
    )

    fold_scores.append(fold_score)
    oof_predictions[valid_index] = valid_predictions

    test_probabilities += model.predict_proba(
        X_test,
        num_iteration=model.best_iteration_
    ) / skf.n_splits

    print(f"Balanced accuracy: {fold_score:.6f}")
    print(f"Best iteration: {model.best_iteration_}")


Fold 1
[100]	valid_0's multi_logloss: 0.189884
[200]	valid_0's multi_logloss: 0.183672
[300]	valid_0's multi_logloss: 0.179324
[400]	valid_0's multi_logloss: 0.175516
[500]	valid_0's multi_logloss: 0.171994
[600]	valid_0's multi_logloss: 0.168639
[700]	valid_0's multi_logloss: 0.165398
[800]	valid_0's multi_logloss: 0.162287
[900]	valid_0's multi_logloss: 0.15941
[1000]	valid_0's multi_logloss: 0.156686
Balanced accuracy: 0.950065
Best iteration: 1000

Fold 2
[100]	valid_0's multi_logloss: 0.190761
[200]	valid_0's multi_logloss: 0.184607
[300]	valid_0's multi_logloss: 0.180473
[400]	valid_0's multi_logloss: 0.176498
[500]	valid_0's multi_logloss: 0.172878
[600]	valid_0's multi_logloss: 0.169416
[700]	valid_0's multi_logloss: 0.166166
[800]	valid_0's multi_logloss: 0.163229
[900]	valid_0's multi_logloss: 0.160359
[1000]	valid_0's multi_logloss: 0.157536
Balanced accuracy: 0.950193
Best iteration: 1000

Fold 3
[100]	valid_0's multi_logloss: 0.193141
[200]	valid_0's multi_logloss: 0.1868

In [11]:
print("Fold scores:", fold_scores)

print("Mean CV balanced accuracy:", np.mean(fold_scores))

print("CV standard deviation:", np.std(fold_scores))

overall_oof_score = balanced_accuracy_score(

    y_encoded,

    oof_predictions

)

print("Overall OOF balanced accuracy:", overall_oof_score)

Fold scores: [np.float64(0.9500651652322044), np.float64(0.9501926496851357), np.float64(0.9488314654218288), np.float64(0.9485572876916084), np.float64(0.9470510335702828)]
Mean CV balanced accuracy: 0.948939520320212
CV standard deviation: 0.0011455627020133543
Overall OOF balanced accuracy: 0.9489395865459117


In [12]:
test_predictions_encoded = np.argmax(
    test_probabilities,
    axis=1
)

test_predictions = label_encoder.inverse_transform(
    test_predictions_encoded
)

print(np.unique(test_predictions, return_counts=True))

(array(['at-risk', 'fit', 'unhealthy'], dtype=object), array([241289,  21171,  33293]))


In [13]:
submission_lgb = sample_submission.copy()

submission_lgb["health_condition"] = test_predictions

submission_lgb.to_csv(
    "submission_lightgbm.csv",
    index=False
)

submission_lgb.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [14]:
print("Submission shape:", submission_lgb.shape)
print(submission_lgb["health_condition"].value_counts())
print(submission_lgb.isnull().sum())

Submission shape: (295753, 2)
health_condition
at-risk      241289
unhealthy     33293
fit           21171
Name: count, dtype: int64
id                  0
health_condition    0
dtype: int64
